Data Prepocessing Step for All 3 Analysis

In [1]:
# !pip install cect (for CENGEN data file)

# import necessary libraries
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import cect
from cect import Cells as C


# import data
EXPRESSION = "../data/021821_stringent_threshold4.csv"   # CeNGEN genes x neuron-class CSV
OUT = Path("inputs")
OUT.mkdir(exist_ok=True)

In [2]:
# Helper Functions

def neuron_to_class(n, classes):
    special = {
        "DA9": "DA9", "VA12": "VA12", "DB1": "DB01", "VB1": "VB01", "VB2": "VB02",
        "VC4": "VC_4_5", "VC5": "VC_4_5", "RMED": "RME_DV", "RMEV": "RME_DV",
        "RMEL": "RME_LR", "RMER": "RME_LR",
    }
    if n in special:
        return special[n]
    if re.fullmatch(r"(DD|VD)\d+", n):
        return "VD_DD"
    if re.fullmatch(r"AS\d+", n):
        return "AS"
    if re.fullmatch(r"DA\d", n):
        return "DA"
    if re.fullmatch(r"DB\d", n):
        return "DB"
    if re.fullmatch(r"VA\d+", n):
        return "VA"
    if re.fullmatch(r"VB\d+", n):
        return "VB"
    if re.fullmatch(r"VC\d", n):
        return "VC"
    if re.fullmatch(r"AWC[LR]", n):
        return "AWC"                      # CeNGEN splits ON/OFF; merged later
    if re.fullmatch(r"IL2[DV][LR]", n):
        return "IL2_DV"
    if re.fullmatch(r"IL2[LR]", n):
        return "IL2_LR"
    if re.fullmatch(r"RMD[DV][LR]", n):
        return "RMD_DV"
    if re.fullmatch(r"RMD[LR]", n):
        return "RMD_LR"
    if n.startswith("CEPsh"):
        return None
    cands = [n]
    if n[-1] in "LR":
        cands.append(n[:-1])
        if len(n) > 2 and n[-2] in "DV":
            cands.append(n[:-2])
    for c in cands:
        if c in classes:
            return c
    return None


def build_types():
    sens = set(C.SENSORY_NEURONS_COOK)
    phar = set(C.PHARYNGEAL_NEURONS)
    motor = (set(C.HEAD_MOTORNEURONS_COOK) | set(C.SUBLATERAL_MOTORNEURONS_COOK)
             | set(C.VENTRAL_CORD_MOTORNEURONS) | set(C.HSN_MOTORNEURONS) | set(C.VC_HERM_MOTORNEURONS))
    inter = set(C.INTERNEURONS_NONPHARYNGEAL_COOK) | set(C.INTERNEURONS_LINK_TO_PHARYNX_COOK)
    return sens, phar, motor, inter

In [3]:
# Expression Matrix

# Assumptions Made: De-Duplicates Gene Names, Merges AWC_ON and AWC_OFF into AWC, Drops Unnamed Columns, Converts Counts to Float

raw = pd.read_csv(EXPRESSION)
raw = raw.drop(columns=[c for c in raw.columns if c.startswith("Unnamed") or c == ""])
classes_all = [c for c in raw.columns if c not in ("gene_name", "Wormbase_ID")]
raw = raw.drop_duplicates("gene_name").set_index("gene_name")
counts = raw[classes_all].astype(float)
awc = counts[["AWC_ON", "AWC_OFF"]].mean(axis=1).rename("AWC")
counts = pd.concat([counts.drop(columns=["AWC_ON", "AWC_OFF"]), awc], axis=1)
classes = set(counts.columns)
print(f"{counts.shape[0]} genes x {counts.shape[1]} classes")
counts.iloc[:5, :6]

13669 genes x 127 classes


,ADA,ADE,ADF,ADL,AFD,AIA
gene_name,,,,,,
nduo-6,7531.007875,12141.685889,3255.836119,3509.352698,6570.565396,4928.395421
ndfl-4,1075.540509,1355.504184,593.604224,534.883641,1379.048087,1319.020788
MTCE.7,1175.681680,1574.857532,159.518954,234.534995,528.557397,552.367659
nduo-1,1732.528365,676.950710,640.879206,1252.851856,2431.787887,1488.218214
atp-6,4470.880547,5169.989040,2648.946349,2080.622062,4449.882959,4065.697703


In [4]:
# Connectome Processing

cache = os.path.join(os.path.dirname(cect.__file__), "cache", "Cook2019HermReader.json")
j = json.load(open(cache))
cook_nodes = j["nodes"]
cs = np.array(j["connections"]["Generic_CS"], float)    # chemical synapses
gj = np.array(j["connections"]["Generic_GJ"], float)    # gap junctions
M = cs + gj
print(f"{len(cook_nodes)} Cook nodes; chemical {int((cs > 0).sum())} nonzero, gap junction {int((gj > 0).sum())} nonzero")

cmap = {n: neuron_to_class(n, classes) for n in cook_nodes}
members = {}
for n, cl in cmap.items():
    if cl:
        members.setdefault(cl, []).append(n)
print(f"{sum(v is not None for v in cmap.values())} nodes mapped to {len(members)} classes; "
      f"{sum(v is None for v in cmap.values())} nodes not mapped (muscles, glia, non-neuronal)")

473 Cook nodes; chemical 4879 nonzero, gap junction 2883 nonzero
301 nodes mapped to 127 classes; 172 nodes not mapped (muscles, glia, non-neuronal)


In [5]:
# Separate by Neuron Type

sens, phar, motor, inter = build_types()
rows = []
for cl, ns in members.items():
    votes = {"pharyngeal": sum(n in phar for n in ns), "sensory": sum(n in sens for n in ns),
             "motor": sum(n in motor for n in ns), "interneuron": sum(n in inter for n in ns)}
    t = max(votes, key=votes.get)
    if votes[t] == 0:
        continue                        # no Cook category (e.g. CAN) -> dropped
    if votes["pharyngeal"] > 0:
        t = "pharyngeal"
    rows.append((cl, t))
meta = pd.DataFrame(rows, columns=["neuron", "type"]).sort_values("neuron")
keep = set(meta.neuron) & set(counts.columns)
meta = meta[meta.neuron.isin(keep)]
meta.type.value_counts()

type
interneuron    45
sensory        39
motor          28
pharyngeal     14
Name: count, dtype: int64

In [6]:
# Edge-List by Neuron Class (mean over member pairs instead of sum)

idx = {n: i for i, n in enumerate(cook_nodes)}
agg = {}
for c1 in keep:
    for c2 in keep:
        if c1 != c2:
            sub = M[np.ix_([idx[n] for n in members[c1]], [idx[n] for n in members[c2]])]
            if sub.sum() > 0:
                agg[(c1, c2)] = sub.mean()        # = sum / (n_c1 * n_c2)

edges = pd.DataFrame([(a, b, w) for (a, b), w in agg.items()],
                     columns=["pre", "post", "weight"])
counts = counts[sorted(keep)]
print(f"{len(counts.columns)} classes, {len(edges)} directed edges")

126 classes, 2332 directed edges


In [7]:
# Write Files and QC

counts.to_csv(OUT / "counts.csv")
meta.to_csv(OUT / "neurons.csv", index=False)
edges.to_csv(OUT / "edges.csv", index=False)
mapping = pd.DataFrame({"class": sorted(keep),
                        "type": meta.set_index("neuron").loc[sorted(keep), "type"].values,
                        "n_members": [len(members[k]) for k in sorted(keep)],
                        "members": [", ".join(members[k]) for k in sorted(keep)]})
mapping.to_csv(OUT / "class_mapping.csv", index=False)

deg = pd.concat([edges.groupby("pre").size(), edges.groupby("post").size()], axis=1).fillna(0).sum(axis=1)
print("classes with no edges:", sorted(set(keep) - set(deg[deg > 0].index)))
print("expression classes dropped (no connectome match / no type):",
      sorted(set(classes) - keep))
print("files written to", OUT.resolve())
mapping

classes with no edges: []
expression classes dropped (no connectome match / no type): ['CAN']
files written to /Users/saisahvirbhaskaruni/Documents/GitHub/Networks_Project/code/inputs


,class,type,n_members,members
0,ADA,interneuron,2,"ADAL, ADAR"
1,ADE,sensory,2,"ADER, ADEL"
2,ADF,sensory,2,"ADFL, ADFR"
3,ADL,sensory,2,"ADLR, ADLL"
4,AFD,sensory,2,"AFDL, AFDR"
...,...,...,...,...
121,VB01,motor,1,VB1
122,VB02,motor,1,VB2
123,VC,motor,4,"VC2, VC1, VC6, VC3"
124,VC_4_5,motor,2,"VC5, VC4"
